# VeriCall — Execution-Verified Post-Training for Tool Calling
### Ground-Up Architecture with In-Context Candidate Schemas & Anti-Collapse DPO

This notebook implements the complete **VeriCall** post-training and evaluation pipeline on **Qwen2.5-7B-Instruct**:
1. **16 Enterprise REST Tool Schemas** (10 Training in Finance/Commerce/CRM, 6 Held-Out Unseen in Cloud Infra/Travel Logistics).
2. **In-Context Candidate Tool Schemas**: Every prompt injects 6 candidate JSON tool schemas (1 target + 5 domain decoys) under `Available Tools:`.
3. **ExecutionValidator**: 6-state error taxonomy (`passed`, `invalid_json`, `wrong_endpoint`, `missing_arg`, `hallucinated_arg`, `type_error`) with markdown fence & regex JSON recovery.
4. **Completion-Only SFT**: Masks prompt tokens so the model only trains on assistant completions (`{"name": "...", "arguments": {...}}`) with `save_strategy="no"`.
5. **Anti-Collapse DPO**: $\beta=0.1$, $\text{lr}=5\times 10^{-6}$, $\text{rpo\_alpha}=0.2$ NLL anchor via `RobustDPOTrainer` on 1,000 near-miss preference pairs.
6. **Pre-Flight Diagnostics**: 25-sample smoke checks after each stage.
7. **Multi-Model Evaluation**: Evaluates Base, Constrained Decoding, SFT, and SFT+DPO across $M=500$ held-out requests with 10,000 bootstrap CIs and paired McNemar tests.


### Stage 1: Environment Setup & Dependencies


In [ ]:
# Install dependencies with unsloth and standard HF stack
import os
import sys
import types
import importlib

print("[*] Installing and verifying dependencies...")
!pip install -q "fastapi>=0.100.0" "pydantic>=2.0.0" "httpx>=0.24.0" "statsmodels>=0.14.0" "scipy>=1.10.0" "tabulate>=0.9.0"

# Clean up broken mocks & safely patch package availability for Python 3.13 / TRL
for pkg in ["llm_blender", "liger_kernel", "liger_kernel.transformers"]:
    if pkg in sys.modules:
        del sys.modules[pkg]

try:
    import transformers.utils.import_utils
    if not getattr(transformers.utils.import_utils, "_is_safe_patched", False):
        _orig_is_pkg = transformers.utils.import_utils._is_package_available
        def _safe_is_pkg(pkg_name, return_version=False):
            if pkg_name in ["liger_kernel", "llm_blender"]:
                return (False, "N/A") if return_version else False
            try:
                return _orig_is_pkg(pkg_name, return_version=return_version)
            except (ValueError, AttributeError):
                return (False, "N/A") if return_version else False
        transformers.utils.import_utils._is_package_available = _safe_is_pkg
        transformers.utils.import_utils._is_safe_patched = True
except Exception:
    pass

# Fix vision mapping in transformers
import transformers.models.auto.modeling_auto
if not hasattr(transformers.models.auto.modeling_auto, "MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES"):
    transformers.models.auto.modeling_auto.MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES = {}

# Neutralize TRL import check flags
try:
    import trl.import_utils
    trl.import_utils.is_llmblender_available = lambda: False
    trl.import_utils.is_liger_kernel_available = lambda: False
except Exception:
    pass

# Attempt unsloth fast install
try:
    import unsloth
    print("[✓] Unsloth is already installed.")
    USE_UNSLOTH = True
except ImportError:
    print("[*] Installing unsloth for 2x faster 4-bit LoRA training...")
    !pip install -q "transformers>=4.40.0" "peft>=0.9.0" "accelerate>=0.28.0" "bitsandbytes>=0.43.0"
    !pip install -q "trl<0.12.0" || pip install -q trl
    try:
        import unsloth
        USE_UNSLOTH = True
        print("[✓] Unsloth installed successfully.")
    except Exception as e:
        print(f"[!] Unsloth install fallback: using Hugging Face PEFT + TRL ({e})")
        USE_UNSLOTH = False

# Verify TRL trainers import cleanly without liger_kernel conflicts
try:
    from trl import SFTTrainer, DPOTrainer, SFTConfig, DPOConfig
    print("[✓] Verified TRL trainers imported cleanly without mock conflicts.")
except Exception as e:
    print(f"[!] Warning on TRL import: {e}")

# Verify and set working directory to repository root (Supports Kaggle, Colab, Local)
from pathlib import Path

if os.path.exists("/kaggle/working"):
    repo_dir = "/kaggle/working/vericall"
    if not os.path.exists(repo_dir):
        print(f"[*] Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    else:
        print(f"[*] Pulling latest changes into {repo_dir}...")
        os.system(f"cd {repo_dir} && git fetch origin && git reset --hard origin/master")
    os.chdir(repo_dir)
elif os.path.exists("/content"):
    repo_dir = "/content/vericall"
    if not os.path.exists(repo_dir):
        print(f"[*] Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/bhuvneshsahu01/vericall.git {repo_dir}")
    else:
        print(f"[*] Pulling latest changes into {repo_dir}...")
        os.system(f"cd {repo_dir} && git fetch origin && git reset --hard origin/master")
    os.chdir(repo_dir)
else:
    curr = Path.cwd()
    if (curr / "pyproject.toml").exists():
        print(f"[✓] Already at repository root: {curr}")
    elif (curr.parent / "pyproject.toml").exists():
        os.chdir(str(curr.parent))
    else:
        print(f"[!] Current directory: {curr}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"[✓] Environment ready. Working directory: {os.getcwd()}")
print(f"[✓] Dependency setup complete. Unsloth engine active: {USE_UNSLOTH}")


### Stage 2: In-Process FastAPI Mock Sandbox & ExecutionValidator (16 Endpoints)


In [ ]:
from vericall.sandbox.validator import ExecutionValidator
from vericall.sandbox.app import app, ENDPOINT_REGISTRY, get_tool_json_schema

validator = ExecutionValidator()
print(f"[✓] Initialized ExecutionValidator with {len(validator.registry)} endpoints across 5 domains.")

# Smoke test all 16 endpoints
test_endpoints = [
    ("issue_refund", {"transaction_id": "txn_12345678", "amount": 49.99, "currency": "USD", "reason": "CUSTOMER_RETURN", "metadata": {"rep": "rep_1"}}),
    ("create_invoice", {"client_id": "cli_1001", "issue_date": "2026-11-01", "due_date": "2026-11-15", "line_items": [{"description": "Dev", "unit_price": 100.0, "quantity": 1}], "currency": "USD"}),
    ("transfer_funds", {"source_account": "acc_100001", "destination_account": "acc_100002", "amount": 250.0, "transfer_method": "ACH", "memo": "Payment"}),
    ("audit_ledger_entries", {"ledger_id": "ledg_100001", "start_date": "2026-10-01", "end_date": "2026-10-31", "entry_type": "CREDIT", "filter_flags": {"reconciled_only": True}}),
    ("query_inventory", {"sku_list": ["SKU-PROD01"], "warehouse_id": "wh_us-01", "include_out_of_stock": False, "fulfillment_channel": "WAREHOUSE_DIRECT"}),
    ("apply_discount_code", {"cart_id": "cart_100001", "promo_code": "SAVE20", "channel": "ONLINE_WEB", "customer_loyalty_pin": 1234}),
    ("update_subscription", {"subscription_id": "sub_10000001", "plan_tier": "PRO_MONTHLY", "auto_renew": True, "addon_features": ["logs"], "effective_date": "2026-12-01"}),
    ("create_support_ticket", {"customer_id": "cust_10001", "priority": "HIGH", "department": "TECHNICAL", "subject": "Platform latency issue", "ticket_context": {"origin": "web"}}),
    ("update_customer_tier", {"customer_id": "cust_10001", "target_tier": "GOLD", "assigned_rep_id": "rep_1001", "override_reason": "High spending account", "benefits": ["waived_fees"]}),
    ("schedule_webhook", {"destination_url": "https://api.partner.com/events", "subscribed_events": ["order.created"], "retry_count": 3, "signing_secret": "whsec_12345678", "headers": {"X-App": "test"}}),
    ("provision_compute_node", {"instance_tier": "c6i.xlarge", "cloud_region": "us-east-1", "cluster_count": 2, "environment_tag": "PRODUCTION", "resource_tags": {"team": "infra"}}),
    ("scale_kubernetes_cluster", {"cluster_name": "k8s-prod-101", "node_pool": "worker-pool", "desired_replicas": 5, "scaling_strategy": "SURGE_FIRST", "autoscale_config": {"min_nodes": 2, "max_nodes": 20}}),
    ("rotate_iam_credentials", {"service_account_id": "sa-data1001", "credential_type": "OAUTH2_KEY", "grace_period_hours": 48, "notify_emails": ["ops@company.com"], "force_revocation": False}),
    ("search_multi_leg_flights", {"itinerary_segments": [{"departure_airport": "JFK", "arrival_airport": "LHR", "flight_date": "2026-11-10"}], "cabin_class": "BUSINESS", "passenger_count": 1, "flexible_dates": False, "preferred_alliances": ["STAR_ALLIANCE"]}),
    ("reserve_corporate_hotel", {"property_code": "HTL-NY-1001", "check_in_date": "2026-12-01", "check_out_date": "2026-12-05", "room_category": "EXECUTIVE_SUITE", "corporate_billing_code": "CORP-10001", "guest_manifest": [{"first_name": "Jane", "last_name": "Doe"}]}),
    ("cancel_itinerary_segment", {"booking_reference": "PNR-ABC123", "segment_id": "seg_01", "cancellation_reason": "CLIENT_RESCHEDULE", "request_credit_voucher": True, "refund_override_notes": "Authorized"}),
]

for fn, args in test_endpoints:
    res = validator.validate_call({"name": fn, "arguments": args}, expected_func=fn)
    assert res.is_success, f"Endpoint {fn} failed: {res.error_message}"

print("[✓] All 16 sandbox endpoints verified functional with HTTP 200 execution.")


### Stage 3: Load Model & Tokenizer (Dual Engine: Unsloth / PEFT 4-Bit)


In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), "ERROR: Kaggle GPU is not enabled! Turn on Accelerator -> GPU T4 x2 in Kaggle Settings."
print(f"[✓] GPU available: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")
import torch
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct"
max_seq_length = 1536

print(f"[*] Loading {model_id} in 4-bit...")

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    print("[✓] Loaded model via unsloth FastLanguageModel.")
else:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    print("[✓] Loaded model via standard HuggingFace BitsAndBytes.")

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[✓] Tokenizer configured (vocab_size={len(tokenizer)}). Ready for evaluation & training.")


### Stage 4: Base Model & Constrained Decoding Evaluation (with 25-Sample Pre-Flight Gate)


In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
import torch
from vericall.eval.evaluate import evaluate_predictions, apply_constrained_decoding

# ── HELPER: Ultra-Fast Batched Generation with Left Padding (batch_size=16, inference_mode) ──
def generate_batched(model, tokenizer, prompts, batch_size=16, max_new_tokens=220, desc="Generating"):
    orig_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Stopping tokens: stop immediately on <|im_end|> so generation stops when JSON closes
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    eos_ids = [tokenizer.eos_token_id] + ([im_end_id] if im_end_id is not None and im_end_id != tokenizer.eos_token_id else [])

    dev = getattr(model, "device", next(model.parameters()).device)
    responses = []

    iterator = range(0, len(prompts), batch_size)
    if len(prompts) > batch_size:
        iterator = tqdm(iterator, desc=desc, total=(len(prompts) + batch_size - 1) // batch_size)

    for i in iterator:
        batch_prompts = prompts[i:i + batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1536,
        ).to(dev)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=eos_ids,
            )

        input_len = inputs.input_ids.shape[1]
        for b in range(len(batch_prompts)):
            gen_tokens = outputs[b][input_len:]
            text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            responses.append(text)

    tokenizer.padding_side = orig_padding_side
    return responses

# 1. Load M=500 unseen API test set (each item has prompt with 6 candidate schemas)
unseen_path = Path("data/eval_unseen_apis.jsonl")
test_records = [json.loads(line) for line in open(unseen_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(test_records)} frozen unseen API evaluation requests.")
assert "Available Tools:" in test_records[0]["prompt"], "Prompt missing candidate tool schemas!"
print(f"[✓] Verified in-prompt candidate schemas present. Target: {test_records[0]['target_function']}")

# 2. Load BFCL AST evaluation requests (N=200)
bfcl_path = Path("data/eval_bfcl_ast_subset.jsonl")
if not bfcl_path.exists():
    bfcl_path = Path("data/eval_bfcl_subset.jsonl")
bfcl_records = [json.loads(line) for line in open(bfcl_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(bfcl_records)} frozen BFCL AST evaluation requests.")

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample Base Model Pre-Flight Smoke Check (batched)...")
smoke_prompts = [item["prompt"] for item in test_records[:25]]
smoke_preds = generate_batched(model, tokenizer, smoke_prompts, batch_size=16, max_new_tokens=220, desc="Base Smoke (25)")
smoke_res = evaluate_predictions(test_records[:25], smoke_preds, "base_smoke")
base_smoke_acc = smoke_res["accuracy_pct"]
print(f"[Base Smoke Check] Passed: {smoke_res['passed_requests']}/25 ({base_smoke_acc}%)")
print(f"                   Errors: {smoke_res['error_taxonomy']}")

# --- FULL 500-SAMPLE EVALUATION (BATCHED) ---
print("\n[*] Generating predictions on full M=500 unseen API test set with Base Model (batched, 8x faster)...")
rest_prompts = [item["prompt"] for item in test_records[25:]]
rest_preds = generate_batched(model, tokenizer, rest_prompts, batch_size=16, max_new_tokens=220, desc="Base Full Eval (475)")
base_predictions = smoke_preds + rest_preds

base_results = evaluate_predictions(test_records, base_predictions, "base_model")
print(f"\n[Base Model] Unseen Accuracy: {base_results['accuracy_pct']}% (Passed: {base_results['passed_requests']}/{base_results['total_requests']})")
print(f"             95% Bootstrap CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"             Errors: {base_results['error_taxonomy']}")

# Evaluate Base + Constrained Decoding (strips markdown fences, repairs truncated braces, coerces types)
print("\n[*] Evaluating Base Model + Constrained Schema Decoding...")
constrained_preds = []
for item, pred in zip(test_records, base_predictions):
    cand_tools = item["candidate_tools"]
    cleaned = apply_constrained_decoding(pred, cand_tools, user_query=item.get("query"))
    constrained_preds.append(cleaned)

constrained_results = evaluate_predictions(test_records, constrained_preds, "base_constrained")
print(f"[Constrained Decoding] Accuracy: {constrained_results['accuracy_pct']}% (Passed: {constrained_results['passed_requests']}/{constrained_results['total_requests']})")
print(f"                      95% CI: [{constrained_results['bootstrap_95_ci'][0]}%, {constrained_results['bootstrap_95_ci'][1]}%]")
print(f"                      Errors: {constrained_results['error_taxonomy']}")

# Evaluate Base on BFCL AST Subset (N=200, realistic 6 candidate tools per prompt)
print("\n[*] Evaluating Base Model on BFCL AST subset (N=200, batched)...")
bfcl_prompts = [item["prompt"] for item in bfcl_records]
bfcl_base_preds = generate_batched(model, tokenizer, bfcl_prompts, batch_size=16, max_new_tokens=220, desc="Base BFCL Eval (200)")
bfcl_base_results = evaluate_predictions(bfcl_records, bfcl_base_preds, "bfcl_base")
bfcl_base_acc_pct = bfcl_base_results["accuracy_pct"]
print(f"[Base Model BFCL] Accuracy: {bfcl_base_acc_pct}% (Passed: {bfcl_base_results['passed_requests']}/{bfcl_base_results['total_requests']})")


### Stage 5: QLoRA Supervised Fine-Tuning (SFT) with Completion-Only Loss Masking


In [ ]:
# Stage 5: QLoRA Supervised Fine-Tuning (SFT) with Completion-Only Loss Masking
import os
import sys
import json
import inspect
from pathlib import Path
import torch

# Clean up broken mocks & safely patch package availability for Python 3.13 / TRL
for pkg in ["llm_blender", "liger_kernel", "liger_kernel.transformers"]:
    if pkg in sys.modules:
        del sys.modules[pkg]

try:
    import transformers.utils.import_utils
    if not getattr(transformers.utils.import_utils, "_is_safe_patched", False):
        _orig_is_pkg = transformers.utils.import_utils._is_package_available
        def _safe_is_pkg(pkg_name, return_version=False):
            if pkg_name in ["liger_kernel", "llm_blender"]:
                return (False, "N/A") if return_version else False
            try:
                return _orig_is_pkg(pkg_name, return_version=return_version)
            except (ValueError, AttributeError):
                return (False, "N/A") if return_version else False
        transformers.utils.import_utils._is_package_available = _safe_is_pkg
        transformers.utils.import_utils._is_safe_patched = True
except Exception:
    pass

try:
    import trl.import_utils
    trl.import_utils.is_llmblender_available = lambda: False
    trl.import_utils.is_liger_kernel_available = lambda: False
except Exception:
    pass

# 1. Bridge transformers Trainer to accept 'tokenizer' from TRL safely
import transformers.trainer

_trainer_sig = inspect.signature(transformers.trainer.Trainer.__init__).parameters
if "tokenizer" not in _trainer_sig and "processing_class" in _trainer_sig:
    if not getattr(transformers.trainer.Trainer.__init__, "_is_patched", False):
        _orig_trainer_init = transformers.trainer.Trainer.__init__
        def _patched_trainer_init(self, *args, **kwargs):
            if "tokenizer" in kwargs and "processing_class" not in kwargs:
                kwargs["processing_class"] = kwargs.pop("tokenizer")
            return _orig_trainer_init(self, *args, **kwargs)
        _patched_trainer_init._is_patched = True
        transformers.trainer.Trainer.__init__ = _patched_trainer_init
        print("[✓] Patched transformers.trainer.Trainer to accept tokenizer as processing_class.")

from datasets import Dataset
from transformers import DataCollatorForLanguageModeling
from trl import SFTTrainer

# Set padding side to right for training stability
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 2. Configure LoRA targeting all linear projections (QLoRA)
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
    if not isinstance(model, PeftModel):
        model = prepare_model_for_kbit_training(model)
        peft_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, peft_config)

# 3. Ensure trainable parameters are in float32 & enable input grads
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[✓] Initialized LoRA adapter. Trainable parameters: {trainable_params:,}")
assert trainable_params > 0, "FATAL: LoRA initialization failed, model has ZERO trainable parameters!"

# 4. Load N=2,000 SFT Training Records & Append <|im_end|> terminal token
sft_train_path = Path("data/train_sft.jsonl")
train_raw = [json.loads(line) for line in open(sft_train_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(train_raw)} training samples from {sft_train_path.name}")

# Format text fields for SFT with terminal token <|im_end|>
train_texts = [item['prompt'] + json.dumps({'name': item['target_function'], 'arguments': item['target_arguments']}) + "<|im_end|>" for item in train_raw]
train_dataset = Dataset.from_dict({"text": train_texts})

# 5. Configure Completion-Only Collator
response_template = "<|im_start|>assistant
"

class RobustCompletionCollator:
    def __init__(self, template: str, tok):
        self.tok = tok
        self.resp_ids = tok.encode(template, add_special_tokens=False)
    def __call__(self, examples):
        texts = [ex["text"] if isinstance(ex, dict) else ex for ex in examples]
        batch = self.tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=1536)
        labels = batch["input_ids"].clone()
        n = len(self.resp_ids)
        for i in range(labels.shape[0]):
            input_ids = batch["input_ids"][i].tolist()
            found = -1
            for j in range(len(input_ids) - n + 1):
                if input_ids[j:j+n] == self.resp_ids:
                    found = j + n
                    break
            if found != -1:
                labels[i, :found] = -100
            else:
                labels[i, :] = -100
        batch["labels"] = labels
        return batch

collator = RobustCompletionCollator(response_template, tokenizer)

# 6. SFT Training Arguments
try:
    from trl import SFTConfig
    cfg_params = inspect.signature(SFTConfig.__init__).parameters
    sft_cfg_kwargs = {
        "output_dir": "checkpoints/runs_sft",
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "num_train_epochs": 1,
        "learning_rate": 2e-4,
        "optim": "adamw_torch",
        "warmup_steps": 10,
        "lr_scheduler_type": "cosine",
        "logging_steps": 20,
        "seed": 42,
        "fp16": False,
        "bf16": False,
        "report_to": "none",
        "dataset_text_field": "text",
        "save_strategy": "no",
    }
    if "max_seq_length" in cfg_params:
        sft_cfg_kwargs["max_seq_length"] = max_seq_length
    elif "max_length" in cfg_params:
        sft_cfg_kwargs["max_length"] = max_seq_length
    sft_args = SFTConfig(**sft_cfg_kwargs)
except ImportError:
    from transformers import TrainingArguments
    sft_args = TrainingArguments(
        output_dir="checkpoints/runs_sft",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        optim="adamw_torch",
        warmup_steps=10,
        lr_scheduler_type="cosine",
        logging_steps=20,
        seed=42,
        fp16=False,
        bf16=False,
        report_to="none",
        save_strategy="no",
    )

# 7. Initialize SFTTrainer & Train
sft_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "data_collator": collator,
    "args": sft_args,
    "tokenizer": tokenizer,
}
sft_trainer = SFTTrainer(**sft_kwargs)
print("[*] Starting QLoRA SFT training with completion-only loss masking...")
sft_trainer.train()

model.save_pretrained("checkpoints/sft_adapter")
tokenizer.save_pretrained("checkpoints/sft_adapter")
print("[✓] SFT training complete. Adapters saved to checkpoints/sft_adapter.")

### Stage 6: Evaluate SFT Model on M=500 Unseen APIs (with Pre-Flight Gate)


In [ ]:
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device

# Restore left padding for generation (Stage 5 used right padding for training)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample SFT Pre-Flight Smoke Check (batched)...")
smoke_sft_prompts = [item["prompt"] for item in test_records[:25]]
smoke_sft_preds = generate_batched(model, tokenizer, smoke_sft_prompts, batch_size=16, desc="SFT Smoke (25)")

smoke_sft_res = evaluate_predictions(test_records[:25], smoke_sft_preds, "sft_smoke")
sft_smoke_acc = smoke_sft_res["accuracy_pct"]
print(f"[SFT Smoke Check] Passed: {smoke_sft_res['passed_requests']}/25 ({sft_smoke_acc}%) vs Base ({base_smoke_acc}%)")
print(f"                  Errors: {smoke_sft_res['error_taxonomy']}")

if sft_smoke_acc < base_smoke_acc:
    print(f"[!] Note: SFT smoke check on 25 samples is {sft_smoke_acc}% (Base: {base_smoke_acc}%). Proceeding to full M=500 evaluation for statistical power.")
else:
    print(f"[✓] PRE-FLIGHT GATE PASSED: SFT accuracy ({sft_smoke_acc}%) matches/exceeds Base ({base_smoke_acc}%).")

# --- FULL 500-SAMPLE EVALUATION (BATCHED) ---
print("\n[*] Generating predictions on full M=500 unseen API test set with SFT Model (batched, 8x faster)...")
rest_sft_prompts = [item["prompt"] for item in test_records[25:]]
rest_sft_preds = generate_batched(model, tokenizer, rest_sft_prompts, batch_size=16, desc="SFT Full Eval (475)")
sft_predictions = smoke_sft_preds + rest_sft_preds

sft_results = evaluate_predictions(test_records, sft_predictions, "sft_only")
print(f"\n[SFT Model] Unseen Accuracy: {sft_results['accuracy_pct']}% (Passed: {sft_results['passed_requests']}/{sft_results['total_requests']})")
print(f"            95% Bootstrap CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"            Errors: {sft_results['error_taxonomy']}")

# --- BFCL AST SUBSET EVALUATION (SFT) ---
print("\n[*] Evaluating SFT Model on BFCL AST subset (N=200, batched)...")
bfcl_sft_preds = generate_batched(model, tokenizer, bfcl_prompts, batch_size=16, desc="SFT BFCL Eval (200)")
bfcl_sft_results = evaluate_predictions(bfcl_records, bfcl_sft_preds, "bfcl_sft")
bfcl_sft_acc_pct = bfcl_sft_results["accuracy_pct"]
print(f"[SFT Model BFCL] Accuracy: {bfcl_sft_acc_pct}% (Passed: {bfcl_sft_results['passed_requests']}/{bfcl_sft_results['total_requests']})")


### Stage 7: Execution-Grounded Near-Miss DPO Mining


In [ ]:
from vericall.data.mine_dpo import mine_dpo_pairs

dpo_pairs = mine_dpo_pairs(num_pairs=1000)
print(f"[✓] Successfully loaded/mined {len(dpo_pairs)} execution-verified preference pairs.")
print("Sample Preference Pair:")
print("  Prompt (first 120 chars):", dpo_pairs[0]["prompt"][:120].replace('\n', ' '))
print("  Chosen:                  ", dpo_pairs[0]["chosen"])
print("  Rejected:                ", dpo_pairs[0]["rejected"], f"({dpo_pairs[0]['rejected_status']})")


### Stage 8: QLoRA Direct Preference Optimization (DPO) with Anti-Collapse Safeguards


In [ ]:
# Stage 8: QLoRA Direct Preference Optimization (DPO) with Anti-Collapse Safeguards
import os
import sys
import json
import inspect
from pathlib import Path
import torch
from datasets import Dataset

# 1. Clean up broken mocks & safely patch package availability
for pkg in ["llm_blender", "liger_kernel", "liger_kernel.transformers"]:
    if pkg in sys.modules:
        del sys.modules[pkg]

try:
    import transformers.utils.import_utils
    if not getattr(transformers.utils.import_utils, "_is_safe_patched", False):
        _orig_is_pkg = transformers.utils.import_utils._is_package_available
        def _safe_is_pkg(pkg_name, return_version=False):
            if pkg_name in ["liger_kernel", "llm_blender"]:
                return (False, "N/A") if return_version else False
            try:
                return _orig_is_pkg(pkg_name, return_version=return_version)
            except (ValueError, AttributeError):
                return (False, "N/A") if return_version else False
        transformers.utils.import_utils._is_package_available = _safe_is_pkg
        transformers.utils.import_utils._is_safe_patched = True
except Exception:
    pass

# Bridge Trainer.__init__ (tokenizer -> processing_class)
import transformers.trainer
_trainer_sig = inspect.signature(transformers.trainer.Trainer.__init__).parameters
if "tokenizer" not in _trainer_sig and "processing_class" in _trainer_sig:
    if not getattr(transformers.trainer.Trainer.__init__, "_is_patched", False):
        _orig_trainer_init = transformers.trainer.Trainer.__init__
        def _patched_trainer_init(self, *args, **kwargs):
            if "tokenizer" in kwargs and "processing_class" not in kwargs:
                kwargs["processing_class"] = kwargs.pop("tokenizer")
            return _orig_trainer_init(self, *args, **kwargs)
        _patched_trainer_init._is_patched = True
        transformers.trainer.Trainer.__init__ = _patched_trainer_init

# 2. Clean Import & Subclass for RobustDPOTrainer
from trl.trainer.dpo_trainer import DPOTrainer as _CleanDPOTrainer
from trl.trainer.dpo_config import DPOConfig

class RobustDPOTrainer(_CleanDPOTrainer):
    """Subclass that handles get_batch_samples, compute_loss, and log signatures cleanly across transformers versions."""

    def get_batch_samples(self, epoch_iterator, num_batches, *args, **kwargs):
        batch_samples = []
        for _ in range(num_batches):
            try:
                batch_samples.append(next(epoch_iterator))
            except StopIteration:
                break
        
        num_items_in_batch = None
        if hasattr(self, "_get_num_items_in_batch"):
            try:
                device = args[0] if len(args) > 0 else kwargs.get("device", getattr(self.args, "device", None))
                num_items_in_batch = self._get_num_items_in_batch(batch_samples, device)
            except Exception:
                num_items_in_batch = None
                
        return batch_samples, num_items_in_batch

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        return _CleanDPOTrainer.compute_loss(self, model, inputs, return_outputs=return_outputs)

    def log(self, logs, *args, **kwargs):
        return _CleanDPOTrainer.log(self, logs)

# 3. Load DPO pairs & ensure <|im_end|> termination
dpo_file = Path("data/train_dpo.jsonl")
dpo_records = [json.loads(line) for line in open(dpo_file, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(dpo_records)} execution-grounded DPO pairs.")

dpo_dict = {
    "prompt": [r["prompt"] for r in dpo_records],
    "chosen": [r["chosen"] if r["chosen"].endswith("<|im_end|>") else r["chosen"] + "<|im_end|>" for r in dpo_records],
    "rejected": [r["rejected"] if r["rejected"].endswith("<|im_end|>") else r["rejected"] + "<|im_end|>" for r in dpo_records],
}
dpo_dataset = Dataset.from_dict(dpo_dict)

# 4. Attach frozen SFT reference adapter to anchor DPO (prevents catastrophic drift)
ref_adapter_loaded = False
if hasattr(model, "load_adapter"):
    try:
        model.load_adapter("checkpoints/sft_adapter", adapter_name="sft_ref")
        if hasattr(model, "set_adapter"):
            model.set_adapter("default")
        ref_adapter_loaded = True
        print("[✓] Successfully loaded reference adapter 'sft_ref' from checkpoints/sft_adapter.\n    DPO is anchored to the SFT model (NOT the untuned base model) with zero extra base VRAM.")
    except Exception as e:
        print(f"[!] Note on loading sft_ref adapter: {e}")

cfg_params = inspect.signature(DPOConfig.__init__).parameters
trainer_params = inspect.signature(_CleanDPOTrainer.__init__).parameters

# 5. Configure DPO Training Arguments (learning_rate: 5e-6 to prevent policy collapse)
dpo_cfg_kwargs = {
    "output_dir": "checkpoints/runs_dpo",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 1,
    "learning_rate": 5e-6,
    "optim": "adamw_torch",
    "warmup_steps": 10,
    "lr_scheduler_type": "cosine",
    "logging_steps": 10,
    "seed": 42,
    "fp16": False,
    "bf16": False,
    "beta": 0.1,
    "report_to": "none",
    "remove_unused_columns": False,
    "max_length": 1536,
    "max_prompt_length": 1280,
    "save_strategy": "no",
    "gradient_checkpointing": False,
}

if "rpo_alpha" in cfg_params:
    dpo_cfg_kwargs["rpo_alpha"] = 0.2
    print("[✓] Enabled RPO NLL anchor (rpo_alpha=0.2) to mathematically prevent policy collapse.")

if ref_adapter_loaded:
    if "ref_adapter_name" in cfg_params:
        dpo_cfg_kwargs["ref_adapter_name"] = "sft_ref"
    if "model_adapter_name" in cfg_params:
        dpo_cfg_kwargs["model_adapter_name"] = "default"
    if "precompute_ref_log_probs" in cfg_params:
        dpo_cfg_kwargs["precompute_ref_log_probs"] = True
        print("[✓] Enabled precompute_ref_log_probs=True against SFT reference adapter.")

dpo_args = DPOConfig(**dpo_cfg_kwargs)

# 6. Ensure default active adapter is selected, cast trainable parameters to float32 & enable input grads
if hasattr(model, "set_adapter"):
    model.set_adapter("default")

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[✓] Active adapter: 'default'. Trainable parameters: {trainable_params:,}")
assert trainable_params > 0, "FATAL: DPO model has ZERO trainable parameters on 'default' adapter!"

# 7. Initialize RobustDPOTrainer & Train
dpo_trainer_kwargs = {
    "model": model,
    "ref_model": None,
    "args": dpo_args,
    "train_dataset": dpo_dataset,
    "tokenizer": tokenizer,
}

if ref_adapter_loaded:
    if "ref_adapter_name" in trainer_params:
        dpo_trainer_kwargs["ref_adapter_name"] = "sft_ref"
    if "model_adapter_name" in trainer_params:
        dpo_trainer_kwargs["model_adapter_name"] = "default"
    if "precompute_ref_log_probs" in trainer_params and "precompute_ref_log_probs" not in cfg_params:
        dpo_trainer_kwargs["precompute_ref_log_probs"] = True

dpo_trainer = RobustDPOTrainer(**dpo_trainer_kwargs)
print("[*] Starting QLoRA Direct Preference Optimization (DPO) with anti-collapse safeguards...")
dpo_trainer.train()

# Ensure default active adapter is selected
if hasattr(model, "set_adapter"):
    model.set_adapter("default")

model.save_pretrained("checkpoints/vericall-dpo-lora")
tokenizer.save_pretrained("checkpoints/vericall-dpo-lora")
model.save_pretrained("checkpoints/dpo_adapter")
tokenizer.save_pretrained("checkpoints/dpo_adapter")
print("[✓] DPO training complete. Adapters saved to checkpoints/vericall-dpo-lora and checkpoints/dpo_adapter.")

### Stage 9: Final Evaluation & Statistical Rigor (Bootstrap CI & McNemar Tests)


In [ ]:
from vericall.eval.stats import compute_mcnemar_test

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()

device = next(model.parameters()).device

# Restore left padding for generation
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

if hasattr(model, "set_adapter"):
    model.set_adapter("default")

# --- 25-SAMPLE PRE-FLIGHT GATE ---
print("\n[*] Running 25-sample DPO Pre-Flight Smoke Check (batched)...")
smoke_dpo_prompts = [item["prompt"] for item in test_records[:25]]
smoke_dpo_preds = generate_batched(model, tokenizer, smoke_dpo_prompts, batch_size=16, desc="DPO Smoke (25)")

smoke_dpo_res = evaluate_predictions(test_records[:25], smoke_dpo_preds, "dpo_smoke")
dpo_smoke_acc = smoke_dpo_res["accuracy_pct"]
print(f"[DPO Smoke Check] Passed: {smoke_dpo_res['passed_requests']}/25 ({dpo_smoke_acc}%) vs SFT ({sft_smoke_acc}%)")
print(f"                  Errors: {smoke_dpo_res['error_taxonomy']}")

if dpo_smoke_acc < sft_smoke_acc:
    print(f"[!] Note: DPO smoke check on 25 samples is {dpo_smoke_acc}% (SFT: {sft_smoke_acc}%). Proceeding to full M=500 evaluation.")
else:
    print(f"[✓] PRE-FLIGHT GATE PASSED: DPO ({dpo_smoke_acc}%) matches/exceeds SFT ({sft_smoke_acc}%).")

# --- FULL 500-SAMPLE EVALUATION (BATCHED) ---
print("\n[*] Generating final predictions with VeriCall (SFT+DPO) on full M=500 unseen APIs (batched, 8x faster)...")
rest_dpo_prompts = [item["prompt"] for item in test_records[25:]]
rest_dpo_preds = generate_batched(model, tokenizer, rest_dpo_prompts, batch_size=16, desc="DPO Full Eval (475)")
dpo_predictions = smoke_dpo_preds + rest_dpo_preds

dpo_results = evaluate_predictions(test_records, dpo_predictions, "sft_dpo_vericall")

# --- BFCL AST SUBSET EVALUATION (VERICALL / DPO) ---
print("\n[*] Evaluating VeriCall (SFT+DPO) on BFCL AST subset (N=200, batched)...")
bfcl_dpo_preds = generate_batched(model, tokenizer, bfcl_prompts, batch_size=16, desc="DPO BFCL Eval (200)")
bfcl_dpo_results = evaluate_predictions(bfcl_records, bfcl_dpo_preds, "bfcl_dpo")
bfcl_dpo_acc_pct = bfcl_dpo_results["accuracy_pct"]
print(f"[VeriCall BFCL]  Accuracy: {bfcl_dpo_acc_pct}% (Passed: {bfcl_dpo_results['passed_requests']}/{bfcl_dpo_results['total_requests']})")

# Fallback checkpoint rollback / warning check:
if dpo_results["accuracy_pct"] < sft_results["accuracy_pct"]:
    print(f"\n[!] WARNING: DPO accuracy ({dpo_results['accuracy_pct']}%) is lower than SFT accuracy ({sft_results['accuracy_pct']}%).")
    print(f"    Possible post-training regression detected. SFT checkpoint is preserved in checkpoints/sft_adapter.")
else:
    print(f"\n[✓] POST-TRAINING ALIGNMENT SUCCESS: DPO accuracy ({dpo_results['accuracy_pct']}%) >= SFT ({sft_results['accuracy_pct']}%).")

# Compute paired McNemar tests
stat_base, p_base = compute_mcnemar_test(base_results["binary_scores"], dpo_results["binary_scores"])
stat_sft, p_sft = compute_mcnemar_test(sft_results["binary_scores"], dpo_results["binary_scores"])

print(f"\n=======================================================")
print(f"FINAL VERICALL BENCHMARK RESULTS (M=500 Unseen APIs)")
print(f"=======================================================")
print(f"[Base Model]     Accuracy: {base_results['accuracy_pct']}%  95% CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"[Base + Constr]  Accuracy: {constrained_results['accuracy_pct']}%  95% CI: [{constrained_results['bootstrap_95_ci'][0]}%, {constrained_results['bootstrap_95_ci'][1]}%]")
print(f"[SFT-Only]       Accuracy: {sft_results['accuracy_pct']}%  95% CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"[VeriCall (DPO)] Accuracy: {dpo_results['accuracy_pct']}%  95% CI: [{dpo_results['bootstrap_95_ci'][0]}%, {dpo_results['bootstrap_95_ci'][1]}%]")
print(f"\nMcNemar Test vs Base:    chi2 = {stat_base:.4f}, p = {p_base:.4e} (Significant: {p_base < 0.05})")
print(f"McNemar Test vs SFT:     chi2 = {stat_sft:.4f}, p = {p_sft:.4e} (Significant: {p_sft < 0.05})")
print(f"-------------------------------------------------------")
print(f"BFCL AST Subset Results (N=200):")
print(f"  Base Model:       {bfcl_base_acc_pct}%")
print(f"  SFT-Only:         {bfcl_sft_acc_pct}%")
print(f"  VeriCall (DPO):   {bfcl_dpo_acc_pct}%")
print(f"=======================================================")


### Stage 10: Save Evaluation Summary & Render Results


In [ ]:
from pathlib import Path
import json
import os
import shutil

summary_data = {
    "metadata": {
        "model_base": model_id,
        "eval_set": "eval_unseen_apis.jsonl",
        "eval_count": len(test_records),
        "bfcl_set": "eval_bfcl_ast_subset.jsonl",
        "bfcl_count": len(bfcl_records),
        "mode": "full_benchmark_v2",
        "device": str(device),
        "num_dpo_pairs": len(dpo_pairs),
    },
    "configs": {
        "base_model": base_results,
        "base_constrained": constrained_results,
        "sft_only": sft_results,
        "sft_dpo_vericall": dpo_results,
    },
    "bfcl_results": {
        "base_model": bfcl_base_results,
        "sft_only": bfcl_sft_results,
        "sft_dpo_vericall": bfcl_dpo_results,
    },
    "bfcl_base_acc_pct": bfcl_base_acc_pct,
    "bfcl_sft_acc_pct": bfcl_sft_acc_pct,
    "bfcl_dpo_acc_pct": bfcl_dpo_acc_pct,
    "mcnemar_tests": {
        "vericall_vs_base": {"statistic": stat_base, "p_value": p_base},
        "vericall_vs_sft": {"statistic": stat_sft, "p_value": p_sft},
    }
}

Path("results").mkdir(exist_ok=True)
with open("results/eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

print("[✓] Saved evaluation summary to results/eval_summary.json")

# Copy to /kaggle/working for direct download
if os.path.exists("/kaggle/working"):
    shutil.copy2("results/eval_summary.json", "/kaggle/working/eval_summary.json")
    print("[✓] Copied to /kaggle/working/eval_summary.json for direct artifact download")

!python scripts/render_results.py
